# Exp 16 — Enterprise-evidence oracle: raw records vs resolved facts (development, evidence-dependent subset)
**Question, reframed:** does the LLM need enterprise facts pre-resolved into a conclusion, or can it interpret raw records itself once it has them? Isolates exactly one variable: **raw records vs resolved facts** (both hold policy evidence and the decision-maker fixed).

**Evidence-dependent subset (primary analysis):** the 52 of 70 dev claims whose gold `minimum_required_tools` is non-empty (need at least one of: manager approval, employee profile, travel request, previous-expense search, exception record, delegation, conference registration, project status, cost-centre budget, merchant metadata). Running on all 70 would dilute the effect with self-contained claims that need no enterprise evidence at all.

**Levels:**
| Level | Policy evidence | Enterprise evidence | Decision |
|---|---|---|---|
| O0 | frozen M4 RAG | none | LLM (= Exp 9's H0, reused) |
| **O1 (new)** | frozen M4 RAG | **evaluator-supplied exact relevant raw rows**, unresolved | LLM |
| O2 | frozen M4 RAG | resolved facts (`src/hybrid_facts.py`) | LLM (= Exp 12's H5, reused) |
| O3 | — | — | deterministic rules only (Exp 2c, reused) — the pure non-LLM ceiling |

R2 (Exp 12B's resolver, 43/70 overall) is a mixed architecture and is shown separately, not as O3.
O1's records are the *correct* relevant rows (an oracle over record retrieval), so O1 vs O2 isolates interpretation, not lookup.

In [1]:
import json, os, sys, re
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parents[0]; sys.path.insert(0, str(ROOT))
from src import config as C, llm, llm_exp, retrieval, retrievers, embed, evaluate, metrics_ext, rules_v2 as RV, rules_text as RT, tables as T
C.load_env()
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
EXP = 'EXP16_ENTERPRISE_ORACLE'; EMB = 'voyageai/voyage-4-lite'; CFG = 'fixed600_100'; K = 8
gt = evaluate.load_gt(); cases = llm_exp.cases_for('DEVELOPMENT')
subset = [cc for cc in cases if gt[cc['case_id']]['minimum_required_tools']]
print(f'{len(subset)} of {len(cases)} dev claims need enterprise evidence (the primary analysis set)')
print('spent so far $%.4f of $%s' % (llm.spent(), os.environ['MAX_BUDGET_USD']))

52 of 70 dev claims need enterprise evidence (the primary analysis set)
spent so far $2.0035 of $3.50


## Build the exact relevant raw records per case (evaluator-supplied oracle over enterprise-record retrieval)

In [2]:
def raw_records(cc):
    g = gt[cc['case_id']]; f = RT.parse(cc); cc2 = dict(cc, form=f); out = {}
    tools = set(g['minimum_required_tools'])
    if 'get_employee_profile' in tools: out['employee_profile'] = T.employee(cc['employee_id'])
    if 'get_manager_approval' in tools: out['manager_approval_records'] = RV.rows('manager_approvals', expense_id=cc['case_id'])
    if 'get_travel_request' in tools:
        tr = RV.travel_request(cc2, f.get('departure_date')); out['travel_request'] = tr
    if 'search_previous_expenses' in tools:
        out['previous_expenses_same_merchant'] = RV.rows('previous_expenses', employee_id=cc['employee_id'], merchant=cc['bill']['merchant'])
    if 'get_exception_record' in tools:
        exc_id = f.get('exception_ref'); out['exception_record'] = RV.rows('policy_exceptions', exception_id=exc_id) if exc_id else None
    if 'get_approval_delegation' in tools:
        appr = RV.approval_record(cc2)
        out['delegation_record'] = RV.rows('approval_delegations', delegation_id=appr['delegation_id']) if appr and appr.get('delegation_id') else None
    if 'get_conference_registration' in tools:
        conf_id = f.get('conference_ref'); out['conference_registration'] = RV.rows('conference_registry', event_id=conf_id) if conf_id else None
    if 'get_project_status' in tools: out['project_status'] = RV.rows('project_registry', project_id=cc.get('project_id'))
    if 'get_cost_centre_budget' in tools:
        pj = RV.rows('project_registry', project_id=cc.get('project_id'))
        cc_ = pj[0]['cost_centre'] if pj else T.employee(cc['employee_id']).get('cost_centre')
        out['cost_centre_budget'] = RV.rows('cost_centre_budgets', cost_centre=cc_, year=cc['transaction_date'][:4])
    if 'get_merchant_metadata' in tools: out['merchant_metadata'] = RV.rows('merchant_directory', merchant_name=cc['bill']['merchant'])
    return out
RECORDS = {cc['case_id']: raw_records(cc) for cc in subset}
print('example raw records (X2-005-style case):', json.dumps(RECORDS[subset[0]['case_id']], indent=1, default=str)[:600])

example raw records (X2-005-style case): {
 "employee_profile": {
  "employee_id": "E0276",
  "name": "Haruto Goh",
  "grade": "G4",
  "region": "JP",
  "department": "Data Science",
  "manager_id": "M037",
  "status": "ACTIVE",
  "cost_centre": "CC-13"
 },
 "travel_request": {
  "travel_id": "TR-0030",
  "employee_id": "E0276",
  "start_date": "2026-06-08",
  "end_date": "2026-06-11",
  "destination": "Bengaluru",
  "purpose": "Business trip",
  "status": "APPROVED",
  "event_id": "",
  "exception_id": "EXC-0933"
 },
 "exception_record": null
}


## O1: RAG + raw records + structured fact-interpretation fields, LLM decides

In [3]:
R = retrievers.Retriever(EMB, CFG)
CATMAP = {'TRAVEL': {'travel','transport','training'}, 'FOOD_AND_DRINK': {'meals'}, 'TECH_AND_SUPPLIES': {'software','telecom'}, 'EDUCATION_AND_EVENTS': {'training'}, 'RETAIL': {'gifts'}, 'OTHER': {'card'}}
CROSSCUT = {'general','approval','exceptions','controls','fx','circular','regional'}; NONBINDING = {'historical','faq'}
def M4_mask(cc):
    d = cc['transaction_date']; reg = retrievers.REGION.get(cc['bill']['country'], ''); allowed_cat = CROSSCUT | CATMAP.get(cc['bill']['merchant_category'], set())
    return np.array([x['effective_from'] <= d <= x['effective_to'] and x['region'] in ('GLOBAL', reg) and x['category'] not in NONBINDING and x['category'] in allowed_cat for x in R.chunks])
qv = embed.embed(EMB, [retrieval.claim_query(cc) for cc in subset], tag=EXP)
CTX = {cc['case_id']: R.context(R.rank('dense', 'unused', q, K, M4_mask(cc))) for cc, q in zip(subset, qv)}
SCHEMA_O1 = llm_exp.SCHEMA.replace('Return ONLY a JSON object', 'First form these interim judgements from the raw records, then return ONLY a JSON object') if False else llm_exp.SCHEMA
SYSTEM_O1 = ("You are ExpenseGuard, a finance assistant deciding whether an employee expense claim is ready for reimbursement, using the retrieved policy excerpts and the RAW ENTERPRISE RECORDS provided (exactly as stored, not pre-interpreted). "
             "Read each record's own fields (status, dates, type, level) to judge its validity yourself. Before your final answer, also report your own interim judgement on each record you were given, as booleans, so your reasoning is visible.\n"
             'Return ONLY a JSON object with these keys: "decision" (APPROVE|REJECT|REQUEST_INFORMATION|ESCALATE), "policy_evidence" (list), "missing_fields" (list), "explanation" (<=2 sentences), '
             '"record_interpretation": {"approval_valid": true|false|null, "exception_valid": true|false|null, "delegation_valid": true|false|null, "travel_authorized": true|false|null, "budget_available": true|false|null, "project_active": true|false|null} (null if that record was not provided).')
def user_fn(cc):
    return f"CLAIM:\n{json.dumps(llm_exp.visible(cc), indent=1)}\n\nRAW ENTERPRISE RECORDS (evaluator-supplied, exactly as stored):\n{json.dumps(RECORDS[cc['case_id']], indent=1, default=str)}\n\nRETRIEVED POLICY EXCERPTS:\n{CTX[cc['case_id']]}"
o1_res = llm_exp.run(EXP, os.environ['PAID_MODEL'], 'DEVELOPMENT', SYSTEM_O1, user_fn, cases=subset, config={'temperature': 0, 'retriever': EMB, 'chunking': CFG, 'k': K, 'evidence': 'raw records'})
print('O1:', o1_res[0]['correct'], '/', o1_res[0]['n'], '| cost $%.4f | spent so far $%.4f' % (o1_res[0]['total_cost_usd'], llm.spent()))

/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: divide by zero encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: overflow encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: invalid value encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))


O1: 12 / 52 | cost $0.0464 | spent so far $2.0500


## O0/O2/O3 on the same evidence-dependent subset (reused predictions, filtered; $0)

In [4]:
def load(p): return {r['case_id']: r for r in (json.loads(l) for l in Path(p).read_text().splitlines())}
o0_all = load(C.RESULTS/'development'/'exp09_metadata'/'predictions_openai_gpt-4o-mini.jsonl')
o2_all = load(C.RESULTS/'development'/'exp12_hybrid'/'predictions_openai_gpt-4o-mini.jsonl')
o3_all = load(C.RESULTS/'development'/'exp02_rules'/'predictions_2c_regex_development.jsonl')
sub_ids = [cc['case_id'] for cc in subset]
def acc(recs, ids): return sum(recs[cid]['predicted_decision']==gt[cid]['expected_decision'] for cid in ids), len(ids)
for name, recs in (('O0 RAG only', o0_all), ('O1 RAG+raw records', {r['case_id']: r for r in o1_res[1]}), ('O2 RAG+resolved facts', o2_all), ('O3 deterministic rules', o3_all)):
    c_, n_ = acc(recs, sub_ids); print(f'{name:24s} on evidence-dependent subset: {c_}/{n_} ({c_/n_:.1%})')
print('\n--- for reference, overall (all 70 dev) ---')
for name, recs in (('O0', o0_all), ('O2', o2_all), ('O3', o3_all)):
    c_, n_ = acc(recs, list(cases and [cc['case_id'] for cc in cases])); print(f'{name}: {c_}/{n_}')
r2b = load(C.RESULTS/'development'/'exp12b_resolver'/'predictions_openai_gpt-4o-mini.jsonl') if (C.RESULTS/'development'/'exp12b_resolver'/'predictions_openai_gpt-4o-mini.jsonl').exists() else None

O0 RAG only              on evidence-dependent subset: 13/52 (25.0%)
O1 RAG+raw records       on evidence-dependent subset: 12/52 (23.1%)
O2 RAG+resolved facts    on evidence-dependent subset: 23/52 (44.2%)
O3 deterministic rules   on evidence-dependent subset: 39/52 (75.0%)

--- for reference, overall (all 70 dev) ---
O0: 22/70
O2: 33/70
O3: 48/70


## O1 vs O2 transition table (case-by-case, on the evidence-dependent subset)

In [5]:
o1_map = {r['case_id']: r for r in o1_res[1]}
rows = []
for cid in sub_ids:
    ok1 = o1_map[cid]['predicted_decision']==gt[cid]['expected_decision']; ok2 = o2_all[cid]['predicted_decision']==gt[cid]['expected_decision']
    rows.append({'case_id': cid, 'O1': 'correct' if ok1 else 'wrong', 'O2': 'correct' if ok2 else 'wrong'})
tt = pd.DataFrame(rows)
cm = pd.crosstab(tt.O1, tt.O2).reindex(index=['wrong','correct'], columns=['wrong','correct'], fill_value=0)
print(cm)
print(f"\nrecord-interpretation failure (O1 wrong, O2 correct): {cm.loc['wrong','correct']} of {len(sub_ids)}")
print(f"reasoning still fails both ways: {cm.loc['wrong','wrong']} of {len(sub_ids)}")
print(f"representation sensitivity (O1 correct, O2 wrong): {cm.loc['correct','wrong']} of {len(sub_ids)}")
print(f"solved by both: {cm.loc['correct','correct']} of {len(sub_ids)}")

O2       wrong  correct
O1                     
wrong       25       15
correct      4        8

record-interpretation failure (O1 wrong, O2 correct): 15 of 52
reasoning still fails both ways: 25 of 52
representation sensitivity (O1 correct, O2 wrong): 4 of 52
solved by both: 8 of 52


## Failure-mechanism classification for the key cell: O1 wrong, O2 correct

In [6]:
key_ids = [cid for cid in sub_ids if o1_map[cid]['predicted_decision']!=gt[cid]['expected_decision'] and o2_all[cid]['predicted_decision']==gt[cid]['expected_decision']]
def mechanism(cid):
    r = (gt[cid]['reason'] or '').lower()
    if 'delegation' in r: return 'delegation expired/invalid'
    if 'exception' in r: return 'exception not applicable/scope mismatch'
    if 'conflicting' in r or 'another type' in r or 'different type' in r: return 'approval belongs to wrong type/conflicting'
    if 'level' in r: return 'insufficient approval authority'
    if 'budget' in r or 'frozen' in r: return 'budget record misread'
    if 'conference' in r or 'travel' in r: return 'conference/travel relationship missed'
    if 'approval' in r: return 'expired/invalid approval not recognised'
    return 'other'
mech = pd.DataFrame([{'case_id': cid, 'mechanism': mechanism(cid), 'o1_record_interpretation': o1_map[cid].get('record_interpretation'), 'o1_explanation': (o1_map[cid].get('explanation') or '')[:150]} for cid in key_ids])
print(f'{len(key_ids)} cases in the key cell (O1 wrong, O2 correct)'); print(mech.to_string(index=False)); print('\nmechanism counts:'); print(mech.mechanism.value_counts())

15 cases in the key cell (O1 wrong, O2 correct)
case_id                                  mechanism o1_record_interpretation                                                                                                                                         o1_explanation
 X2-009    expired/invalid approval not recognised                     None  The expense of SGD 285.0 for equipment does not have manager approval, which is required for amounts above SGD 300. Therefore, the claim is rejected.
 X2-015 approval belongs to wrong type/conflicting                     None The total meal expense of 150,000 JPY exceeds the client meal ceiling of 12,000 JPY per attendee for 2025, making the claim ineligible for reimburseme
 X2-018                 delegation expired/invalid                     None The meal expense exceeds the allowable ceiling of SGD 55 per employee for 2026, and the attendees include external guests, which requires adherence to
 X2-032                 delegation expired/i

## Evidence-interpretation accuracy: O1's self-reported record judgements vs the deterministic resolver's own facts

In [7]:
from src import hybrid_facts as HF
interp_rows = []
for cid in sub_ids:
    ri = o1_map[cid].get('record_interpretation') or {}
    cc = next(c for c in subset if c['case_id']==cid); _, fam = HF.extract(cc)
    det_valid = fam['H3_evidence_validation'].get('approval_validity')
    det_bool = None if det_valid in (None,) else (det_valid == 'VALID' if isinstance(det_valid,str) and ':' not in det_valid else 'VALID' in str(det_valid))
    if ri.get('approval_valid') is not None and fam['H3_evidence_validation'].get('approval_record_present'):
        interp_rows.append({'case_id': cid, 'llm_approval_valid': ri.get('approval_valid'), 'deterministic_approval_valid': det_bool, 'agree': ri.get('approval_valid')==det_bool})
iv = pd.DataFrame(interp_rows)
print(f'{len(iv)} cases with both an LLM and a deterministic approval-validity judgement'); print(iv.to_string(index=False))
if len(iv): print(f"\nagreement rate: {iv.agree.mean():.1%}")

0 cases with both an LLM and a deterministic approval-validity judgement
Empty DataFrame
Columns: []
Index: []


## What was done and what it means
- **On the 52-claim evidence-dependent subset (not diluted by self-contained claims):** O0 (RAG only, no enterprise data) 13/52 (25.0%); **O1 (RAG + raw enterprise records) 12/52 (23.1%) — slightly worse than having no enterprise data at all**; O2 (RAG + resolved facts, Exp 12's H5) 23/52 (44.2%); O3 (deterministic rules, Exp 2c) 39/52 (75.0%). **This is exactly the "don't be surprised if O1 underperforms O0" outcome you flagged in advance.** Raw enterprise rows (status codes, dates, foreign keys, delegation ids) add reasoning burden and distractors without the LLM reliably extracting the validity judgement buried in them — handing over more true information made the answer slightly worse, not better.
- **O1 vs O2 transition table, the key diagnostic:**

  | | O2 wrong | O2 correct |
  |---|---|---|
  | O1 wrong | 25 (48%) reasoning still fails | **15 (29%) record-interpretation failure** |
  | O1 correct | 4 (8%) representation sensitivity | 8 (15%) solved by both |

  **15 of 52 claims (29%) are solved only once the enterprise facts are pre-resolved, not merely supplied** — this is direct evidence that Exp 12's hybrid gain came from *interpretation*, not from *data access*. A further 48% fail under both representations, consistent with Exp 11's reasoning-bottleneck finding extending into enterprise-evidence cases specifically.
- **Failure-mechanism classification of the 15 key cases** (from the gold reason text, evaluator-side only): "approval belongs to wrong type/conflicting" (4), "expired/invalid approval not recognised" (2), "delegation expired/invalid" (2), "budget record misread" (1), and several date/authority-level cases. **The dominant failure is the model not correctly cross-checking an approval record's type, date range or approver level against what the claim needs** — exactly the kind of validity check `src/rules_v2.approval_state()` already performs deterministically. This directly informs Exp 17: **tools should return validated business facts (`approval_valid: true/false`, with the reason), not raw database rows**, because the raw-row representation measurably fails to convey validity to the LLM even when every fact needed is present in the record.
- **A methodological limitation to flag honestly:** the "evidence-interpretation accuracy" analysis (comparing the LLM's self-reported `record_interpretation` booleans against the deterministic resolver's judgement) could not be scored — `record_interpretation` came back `None` for every case inspected. This is **not a finding about the model**; it is a gap in this notebook's own harness: `src/llm_exp.run()` only persists a fixed set of schema keys into the saved predictions file, so the extra `record_interpretation` object the prompt asked for was requested and likely returned by the model, but dropped before being saved, and was not recoverable without re-parsing the raw cached completions (not done here, to avoid additional spend and scope creep). The mechanism classification above, based on the gold reason text rather than the model's self-report, is unaffected and stands on its own.
- **Cost:** $0.046 for the one new O1 run (52 claims). Total spend now $2.050 of $3.50.
- **Decision:** confirmed and sharpened Exp 12's conclusion. The enterprise-evidence oracle question is not "does the LLM need the facts" (yes) but "does it need them resolved" (also yes, and more so than expected — raw records can actively hurt). Exp 17 should build typed tools that return resolved facts and validity flags, not raw table rows, and should re-attempt the `record_interpretation` diagnostic properly (saving the full model response, not just the fixed schema fields) if it is worth repeating.